# Residual CO2 emissions by sector in 1.5 °C pathways (AR6 C1)

**Question.** How large are residual gross CO2 emissions by sector in 2050 in AR6 C1 pathways (1.5 °C, no or limited overshoot), and how much do they differ across model families?

**Data.** AR6 Scenarios Database hosted by IIASA, v1.1 (doi:10.5281/zenodo.5886911), World snapshot and metadata indicators, placed in `data/raw/` (see `data/raw/README.md`).

The notebook runs top to bottom. Helper functions are in `src/ar6_residuals.py`.

In [1]:
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pyam

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import ar6_residuals as ar

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"
for d in (PROCESSED, RESULTS, FIGURES):
    d.mkdir(exist_ok=True)

CSV = RAW / "AR6_Scenarios_Database_World_v1.1.csv"
META = RAW / "AR6_Scenarios_Database_metadata_indicators_v1.1.xlsx"
YEAR = 2050
FAMILIES = ["MESSAGEix-GLOBIOM", "REMIND-MAgPIE", "IMAGE"]

logging.getLogger("pyam").setLevel(logging.ERROR)  # silence 'ignoring meta' notices
pd.set_option("display.width", 200)
print("pyam", pyam.__version__, "| pandas", pd.__version__)

pyam 3.5.0 | pandas 2.3.3


## 1. Load and filter

The World file holds about 690 000 timeseries. Only the 22 CO2 emission and carbon-sequestration variables listed in `ar.ALL_VARIABLES` and the year 2050 are read; pandas does this pre-filtering, then the result becomes a `pyam.IamDataFrame`. The metadata sheet `meta` supplies the climate category, which is used as a filter like any data dimension.

In [2]:
df = ar.load_ar6(CSV, META, years=(YEAR,))
c1 = df.filter(Category="C1", region="World", year=YEAR)
print(f"{len(df.index)} scenarios with at least one selected variable; {len(c1.index)} in C1")
c1

1919 scenarios with at least one selected variable; 97 in C1


<class 'pyam.core.IamDataFrame'>
Index:
 * model    : AIM/CGE 2.1, AIM/CGE 2.2, AIM/Hub-Global 2.0, ... WITCH-GLOBIOM 4.4 (24)
 * scenario : CD-LINKS_NPi2020_400, EN_NPi2020_300f, ... CD-LINKS_NPi2020_1000 (78)
Timeseries data coordinates:
   region   : World (1)
   variable : Carbon Sequestration|CCS|Biomass, ... Emissions|CO2|Other (22)
   unit     : Mt CO2/yr (1)
   year     : 2050 (1)
Meta indicators:
   Category (object) C1 (1)
   Category_name (object) ... (1)
   Category_subset (object) C1a_NZGHGs, C1b_+veGHGs (2)
   Subset_Ch4 (object) ... (1)
   Category_Vetting_historical (object) C1 (1)
   ...

### Model families

A model family is the model name without its version (`REMIND-MAgPIE 2.1-4.2` → `REMIND-MAgPIE`). Versions of one model share structure and assumptions, so they are grouped. `MESSAGE-GLOBIOM 1.0` is kept separate from `MESSAGEix-GLOBIOM`, following the rule strictly.

In [3]:
c1.set_meta(c1.meta.index.get_level_values("model").map(ar.model_family), name="family")
fam_counts = c1.meta.groupby("family").size().sort_values(ascending=False).rename("C1 scenarios")
fam_counts.to_frame().T

family,REMIND-MAgPIE,MESSAGEix-GLOBIOM,REMIND,IMAGE,GCAM,WITCH,MESSAGE-GLOBIOM,WITCH-GLOBIOM,AIM/CGE,POLES EMF33,C-ROADS-5.005,GEM-E3_V2021,AIM/Hub-Global,COFFEE,POLES ADVANCE
C1 scenarios,28,16,13,7,6,5,4,4,3,3,3,2,1,1,1


## 2. Reporting coverage

Not every scenario reports every variable. The table counts C1 scenarios with a 2050 value per variable for the three families compared later.

In [4]:
cov = (
    c1.filter(family=FAMILIES).data
    .assign(family=lambda d: d.model.map(ar.model_family))
    .pivot_table(index="variable", columns="family", values="value", aggfunc="size")
    .reindex(ar.ALL_VARIABLES).fillna(0).astype(int)
)
cov.loc["(C1 scenarios)"] = fam_counts[FAMILIES]
cov

family,IMAGE,MESSAGEix-GLOBIOM,REMIND-MAgPIE
variable,,,
Carbon Sequestration|CCS|Biomass,7,16,28
Carbon Sequestration|CCS|Biomass|Energy|Demand|Industry,7,0,26
Carbon Sequestration|CCS|Biomass|Energy|Supply,7,9,26
Carbon Sequestration|CCS|Biomass|Industrial Processes,7,0,0
Carbon Sequestration|CCS|Fossil,7,15,17
Carbon Sequestration|CCS|Industrial Processes,7,14,21
Carbon Sequestration|Direct Air Capture,0,0,17
Carbon Sequestration|Enhanced Weathering,0,0,17
Carbon Sequestration|Land Use,7,16,24


## 3. Consistency check: do components add up to the reported totals?

Four aggregation rules are checked (`ar.AGGREGATION_RULES`):

| parent | required components | optional |
|---|---|---|
| `Emissions\|CO2` | Energy and Industrial Processes, AFOLU | Other |
| `Emissions\|CO2\|Energy and Industrial Processes` | Energy\|Supply, Energy\|Demand | Industrial Processes |
| `Emissions\|CO2\|Energy\|Demand` | Industry, Transportation, Residential and Commercial | AFOFI, Other Sector |
| `Carbon Sequestration\|CCS\|Biomass` | Energy\|Supply | Energy\|Demand\|Industry, Industrial Processes |

A scenario that lacks a required component is counted as *incomplete* rather than *inconsistent*; a missing child would otherwise look like a large error. Complete scenarios fail if |parent − sum| > 10 Mt CO2/yr + 1 % of the sum. `pyam.require_data` finds the incomplete ones and `pyam.check_aggregate` does the comparison.

In [5]:
summary, failures = ar.check_aggregates(c1, rtol=0.01, atol=10.0)
failures.insert(1, "family", failures["model"].map(ar.model_family))
failures.to_csv(RESULTS / "consistency_failures_C1_2050.csv", index=False)
summary

,reported,incomplete,checked,failed
parent,,,,
Emissions|CO2,97,5,92,5
Emissions|CO2|Energy and Industrial Processes,92,7,85,19
Emissions|CO2|Energy|Demand,90,15,75,7
Carbon Sequestration|CCS|Biomass,95,28,67,14


In [6]:
(failures.groupby(["parent", "family"])["difference"]
 .agg(n="size", median="median", min="min", max="max").round(0))

n   median      min      max
parent                                        family                                         
Carbon Sequestration|CCS|Biomass              GCAM               1     51.0     51.0     51.0
                                              IMAGE              7  -1499.0  -3341.0  -1249.0
                                              REMIND             1   -441.0   -441.0   -441.0
                                              REMIND-MAgPIE      5   -465.0   -499.0   -293.0
Emissions|CO2                                 AIM/CGE            2     92.0     92.0     93.0
                                              REMIND-MAgPIE      3    278.0    278.0    281.0
Emissions|CO2|Energy and Industrial Processes AIM/CGE            1   1543.0   1543.0   1543.0
                                              GCAM               1  16240.0  16240.0  16240.0
                                              MESSAGE-GLOBIOM    3   1664.0    241.0   1686.0
                                              MESSAGEix-GLOBIOM  1   2382.0   2382.0   2382.0
                                              POLES ADVANCE      1   1001.0   1001.0   1001.0
                                              REMIND             1    278.0    278.0    278.0
                                              REMIND-MAgPIE      6   -939.0  -1247.0   1407.0
                                              WITCH              5   -530.0   -900.0   -483.0
Emissions|CO2|Energy|Demand                   MESSAGEix-GLOBIOM  7   1388.0   1315.0   1842.0

Main patterns (Mt CO2/yr, 2050):

* **IMAGE**, `CCS|Biomass`: the parent equals supply-side BECCS only; industrial BECCS (≈1.2–3.3 Gt) is reported as a child but left out of the parent. The analysis therefore takes BECCS as the larger of the parent and the sum of its children.
* **MESSAGEix-GLOBIOM**, `Energy|Demand`: in 7 scenarios the demand total exceeds the sum of industry, transport and buildings by 1.3–1.8 Gt; part of demand-side emissions is not allocated to a reported sector.
* **WITCH** and **REMIND-MAgPIE**, `Energy and Industrial Processes`: the total is mostly below the sector sum (median −0.5 and −0.9 Gt); the reported variables do not show which term causes the gap.

The full list of failing scenarios is written to `results/consistency_failures_C1_2050.csv`.

In [7]:
failures.sort_values(["parent", "family", "scenario"]).reset_index(drop=True)

,model,family,scenario,parent,parent_value,sum_of_components,difference
0,GCAM 4.2,GCAM,SSP1-19,Carbon Sequestration|CCS|Biomass,2339.605000,2288.154000,51.451000
1,IMAGE 3.2,IMAGE,SSP1_SPA1_19I_D_LB,Carbon Sequestration|CCS|Biomass,3450.764943,4702.336877,-1251.571934
2,IMAGE 3.2,IMAGE,SSP1_SPA1_19I_LIRE_LB,Carbon Sequestration|CCS|Biomass,4055.792219,5706.785487,-1650.993268
3,IMAGE 3.2,IMAGE,SSP1_SPA1_19I_RE_LB,Carbon Sequestration|CCS|Biomass,3457.139484,4798.882785,-1341.743300
4,IMAGE 3.2,IMAGE,SSP2_SPA1_19I_D_LB,Carbon Sequestration|CCS|Biomass,3534.076421,4782.627442,-1248.551021
5,IMAGE 3.2,IMAGE,SSP2_SPA1_19I_LIRE_LB,Carbon Sequestration|CCS|Biomass,3691.713729,5190.777996,-1499.064267
6,IMAGE 3.2,IMAGE,SSP2_SPA1_19I_RE_LB,Carbon Sequestration|CCS|Biomass,3833.937775,5365.276358,-1531.338583
7,IMAGE 3.2,IMAGE,SSP2_SPA2_19I_LI,Carbon Sequestration|CCS|Biomass,9281.331052,12621.846925,-3340.515872
8,REMIND 1.7,REMIND,ADVANCE_2020_1.5C-2100,Carbon Sequestration|CCS|Biomass,7975.377000,8416.028700,-440.651700
9,REMIND-MAgPIE 1.7-3.0,REMIND-MAgPIE,CD-LINKS_NPi2020_400,Carbon Sequestration|CCS|Biomass,7606.074000,8071.202300,-465.128300


## 4. Gross sectoral emissions and CDR in 2050

AR6 sectoral variables are **net** of carbon capture in the sector, and BECCS is a negative entry inside energy supply (and, in some models, industry). Gross residual emissions per sector are therefore net sectoral emissions plus the BECCS captured in that sector (`ar.sector_table`, rules in its docstring). Fossil CCS is not added back: it prevents emissions but does not remove CO2.

AFOLU stays **net**: most models report only the net land-use flux, and adding `Carbon Sequestration|Land Use` back does not give a consistent gross value (it turns negative for some REMIND-MAgPIE scenarios).

Where net energy supply is negative and a model reports no BECCS at all (two GEM-E3 scenarios), the gross value cannot be derived and is left missing.

In [8]:
tab = ar.sector_table(c1, year=YEAR)
tab.to_csv(PROCESSED / "c1_2050_sectors_cdr.csv", index=False)   # not under version control (licence)
print(f"{len(tab)} scenarios; BECCS fully attributed to supply in {tab.beccs_supply_assumed.sum()} of them")
cols = ar.SECTORS + ["Gross residual (4 sectors)"] + ar.CDR_COMPONENTS
tab[tab.family.isin(FAMILIES)].groupby("family")[cols].median().round(0).T

97 scenarios; BECCS fully attributed to supply in 28 of them


family,IMAGE,MESSAGEix-GLOBIOM,REMIND-MAgPIE
Energy supply,1264.0,748.0,1314.0
Industry,3870.0,1350.0,2074.0
Transport,1655.0,2615.0,3643.0
Buildings,726.0,211.0,578.0
AFOLU (net),-3923.0,-2663.0,78.0
Gross residual (4 sectors),7816.0,5567.0,7181.0
BECCS,5191.0,1000.0,4087.0
DACCS,NaN,NaN,24.0
Land-based removal,4088.0,3791.0,236.0
Enhanced weathering,NaN,NaN,0.0


## 5. Comparison across model families

Summary statistics are medians and inter-quartile ranges over scenarios. No significance tests are applied: AR6 is an ensemble of opportunity, not a random sample, and scenarios within a family often come from the same study (all seven IMAGE scenarios are from one study).

In [9]:
stats = ar.summary_stats(tab, FAMILIES)
stats.to_csv(RESULTS / "summary_C1_2050.csv")
stats["median"].unstack("group")[["All C1"] + FAMILIES].loc[ar.SUMMARY_COLUMNS]

group,All C1,MESSAGEix-GLOBIOM,REMIND-MAgPIE,IMAGE
quantity,,,,
Energy supply,1.29,0.75,1.31,1.26
Industry,1.94,1.35,2.07,3.87
Transport,3.10,2.61,3.64,1.66
Buildings,0.59,0.21,0.58,0.73
Gross residual (4 sectors),7.33,5.57,7.18,7.82
AFOLU (net),-2.57,-2.66,0.08,-3.92
BECCS,4.17,1.00,4.09,5.19
DACCS,0.10,NaN,0.02,NaN
Land-based removal,3.44,3.79,0.24,4.09


In [10]:
stats.loc["All C1"]

,n,p25,median,p75
quantity,,,,
Energy supply,87,0.83,1.29,1.59
Industry,90,1.44,1.94,3.05
Transport,85,2.36,3.10,4.57
Buildings,79,0.37,0.59,0.87
Gross residual (4 sectors),68,5.81,7.33,9.20
AFOLU (net),97,-2.92,-2.57,-0.07
BECCS,95,1.93,4.17,5.45
DACCS,38,0.00,0.10,0.99
Land-based removal,64,0.32,3.44,4.09


Study clustering within each family:

In [11]:
(c1.meta[c1.meta.family.isin(FAMILIES)]
 .groupby(["family", "Project_study"]).size().rename("scenarios").to_frame())

scenarios
family            Project_study             
IMAGE             van Vuuren 2021          7
MESSAGEix-GLOBIOM CD-LINKS                 1
                  ENGAGE                   7
                  Grubler 2018             1
                  Kikstra 2021             5
                  NGFS2                    2
REMIND-MAgPIE     Bertram 2018             1
                  CD-LINKS                 1
                  EMF33                    1
                  ENGAGE                   8
                  Kriegler 2018            3
                  Luderer 2021             2
                  NGFS2                    4
                  SSP                      1
                  Soergel 2021             3
                  Strefler 2021b           4

## 6. Figures

Values in Gt CO2/yr. Jitter uses a fixed random seed.

In [12]:
import matplotlib.pyplot as plt
import plots

plots.set_style()
fig1 = plots.fig_sector_distribution(tab)
fig1.savefig(FIGURES / "fig1_sector_distribution_C1_2050.png")

**Figure 1.** Gross residual CO2 emissions by sector in 2050 across all C1 scenarios that report the sector (n above each column). Energy supply and industry include BECCS added back; AFOLU is the net land-use flux. Boxes: inter-quartile range; whiskers: 5th–95th percentile; points: individual scenarios.

In [13]:
fig2 = plots.fig_family_comparison(tab)
fig2.savefig(FIGURES / "fig2_family_comparison_C1_2050.png")

**Figure 2.** (a) Sectoral CO2 in 2050 for the three model families; bars mark family medians. Legend n is the number of C1 scenarios per family; not every scenario reports every sector. (b) Sum of the four energy and industry sectors, only for scenarios that report all four. For MESSAGEix-GLOBIOM about 1.5 Gt of demand-side emissions are not allocated to a sector (Section 3) and are therefore missing from this sum.

In [14]:
fig3 = plots.fig_balance(tab)
fig3.savefig(FIGURES / "fig3_balance_by_scenario_C1_2050.png")

**Figure 3.** Per-scenario CO2 balance in 2050, sorted by gross residual emissions within each family. Positive bars: gross sectoral emissions; negative bars: net AFOLU sink, BECCS, DACCS and enhanced weathering. Land-based removal is contained in net AFOLU and is not drawn separately. The hatched segment is the part of the reported net CO2 (dots) that the stacked components do not explain; it is largest for MESSAGEix-GLOBIOM (median ≈ 1.5 Gt).

## 7. Results

* **Size.** Across C1 scenarios the four energy and industry sectors still emit a median of **7.3 Gt CO2/yr** in 2050 (inter-quartile range 5.8–9.2, n = 68). Transport (median 3.1) and industry (1.9) make up about three quarters of this; energy supply (1.3) and buildings (0.6) the rest.
* **Model families.** Totals differ less than their composition. IMAGE has the largest industrial residual (median 3.9 Gt, about three times MESSAGEix-GLOBIOM), REMIND-MAgPIE the largest transport residual (3.6 Gt, about twice IMAGE). The lower MESSAGEix-GLOBIOM total (5.6 Gt) is partly a reporting artefact: about 1.5 Gt of its demand-side emissions are not allocated to a sector.
* **Balancing.** REMIND-MAgPIE balances residual emissions almost entirely with BECCS (median 4.1 Gt) and a near-zero net AFOLU flux; IMAGE combines BECCS (5.2 Gt) with a large land sink (net AFOLU −3.9 Gt); MESSAGEix-GLOBIOM relies mainly on the land sink (−2.7 Gt) with little BECCS (1.0 Gt). DACCS is reported by few scenarios and is small in 2050 (median 0.1 Gt, n = 38).
* **Spread within families** is comparable to the spread between them: REMIND-MAgPIE gross residuals range from 3.3 to 18.5 Gt, depending on study and scenario design.